# 05 – Nevralt nettverk, evaluering og forklaring

**Mål:** trene et nettverk med ett skjult lag, forklare tensorformene og
undersøke feil på nye bilder.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import CLASS_NAMES, download_fashion_mnist

train_data, test_data = download_fashion_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


## Eksempel: et skjult lag og ReLU

I notatbok 04 brukte vi ett lineært lag fra 784 piksler til ti skårer.
Nå legger vi inn et skjult lag med ReLU før det siste lineære laget.
Treningsløkken fra 04 kan brukes også med denne modellen.

Et skjult lag lager mellomliggende representasjoner. ReLU erstatter negative
verdier med 0 og beholder positive. Flere lineære lag uten aktiveringsfunksjon mellom seg kan slås sammen til
ett lineært lag. Flere lag alene gjør derfor ikke modellen i stand til å
representere en ikke-lineær funksjon. Med konstantledd er beregningen strengt
tatt *affin*, slik som en rett linje som ikke trenger å gå gjennom origo.

Her er et lite eksempel med tre innganger, fire skjulte nevroner og to utganger.

*Merk*: Det siste laget gir rå skårer, og vi bruker ingen aktiveringsfunksjon etter det.


In [ ]:
small_model = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 2))
print(small_model(torch.ones(5, 3)).shape)
values = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("Før ReLU:", values, "Etter:", nn.ReLU()(values))


### Nettverket som graf

![Et fullkoblet nettverk med tre innganger, fire skjulte nevroner med ReLU og to utganger.](../assets/05_nettverk.svg)

Figuren viser den lille modellen over, for **ett eksempel**. Alle innganger er
koblet til alle nevronene i neste lag. Hver forbindelse har en vekt.
Et skjult nevron beregner en vektet sum pluss et konstantledd, og bruker så ReLU.
Utgangene beregner vektede summer pluss konstantledd, uten ReLU.


### Nøtt: Hvorfor trenger nevrale nettverk aktiveringsfunksjoner?

Et nevralt nettverk består av flere lag som bearbeider informasjon.
Uten aktiveringsfunksjoner kan flere lineære lag slås sammen til ett lineært lag.
Men hva betyr det for hva nettverket kan lære?

Anta at vi ønsker å trene et nettverk til å tilnærme funksjonen $f(x) = x^2$
for $x$ mellom $-1$ og $1$.

![Til venstre: En rett linje treffer ikke alle tre markerte punktene på parabolen. Til høyre: Flere rette linjestykker følger parabolen tettere.](../assets/05_aktivering.svg)

*Figuren viser to håndlagde eksempler, ikke resultater fra trening.
Med ReLU mellom lagene kan nettverket lage en graf med knekkpunkter.*

1. **Lineære lag:** Hvorfor hjelper det ikke å legge til flere lineære lag
   uten aktiveringsfunksjoner når vi vil tilnærme $x^2$?
   Bruk to enkle lag, $h = ax + b$ og $y = ch + d$:
   Sett inn uttrykket for $h$ og vis at resultatet fortsatt er en rett linje,
   $y = Ax + B$. Kan én rett linje treffe alle punktene $(-1, 1)$, $(0, 0)$ og $(1, 1)$?
2. **Aktiveringsfunksjoner:** ReLU er gitt ved $\mathrm{ReLU}(z) = \max(0, z)$.
   Hvordan gjør denne funksjonen det mulig å lage knekkpunkter og tilnærme
   en buet graf? Bruk figuren til å forklare med egne ord.

<details>
<summary>Hint</summary>

Start med $y = c(ax + b) + d$ og gang ut parentesen.
Hva er stigningstallet, og hva er konstantleddet?
Med konstantledd kalles dette strengt tatt en *affin* funksjon;
her bruker vi «lineært lag» slik som i `nn.Linear`.

ReLU er null når inngangen er negativ og lik inngangen ellers.
Grafen har derfor en knekk ved null. I $\mathrm{ReLU}(ax + b)$ kan vekten $a$
og konstantleddet $b$ endre hvor knekken ligger og hvilken vei grafen heller.
Flere slike bidrag kan summeres i neste lag.

</details>


**Mitt svar:**

Skriv din forklaring her.


## Oppgave 1: Lag Fashion-MNIST-nettverket

Skriv funksjonen `create_model(hidden_size)` som returnerer følgende modell (nettverk):
`Flatten → Linear(784, hidden_size) → ReLU → Linear(hidden_size, 10)`.

**Regn for hånd før du skriver kode.** Bruk 128 skjulte nevroner.
Hvor mange vekter og konstantledd har det første lineære laget?
Hvor mange har det siste? Har `Flatten` eller `ReLU` parametere som læres?
Skriv utregningen og totalen i svarcellen under. Bruk antall forbindelser
inn til hvert nevron som hjelp; ikke bruk `demo_model.parameters()` ennå.

Opprett deretter `demo_model = create_model(128)`. Vis formen etter hvert lag for en gruppe
på 32 nullbilder. Tell parametrene med kode og sammenlign med håndregningen.

<details>
<summary>Hint til koden</summary>

Bruk en vanlig funksjon som returnerer `nn.Sequential`. Iterer over lagene for å vise mellomformene.
Tell parametrene med `sum(parameter.numel() for parameter in demo_model.parameters())`.

</details>


**Mitt svar:**

Skriv utregningen din her.


In [ ]:
# Lag funksjonen create_model(hidden_size) som returnerer modellen.
# Skriv din kode her.


**Kontroll:** Kjør cellen for å sjekke løsningen.


In [ ]:
assert demo_model(torch.zeros(32, 1, 28, 28)).shape == (32, 10), "Modellen skal gi formen (32, 10) for 32 bilder. Sjekk lagene og antall utverdier."

assert sum(p.numel() for p in demo_model.parameters()) == 101770, "Nettverket skal ha 101770 parametere. Sjekk størrelsene på de to Linear-lagene."
assert demo_model(torch.zeros(1, 1, 28, 28)).shape == (1, 10), "Modellen skal også fungere for ett bilde og gi formen (1, 10)."

print("Kontroll bestått!")


### Refleksjon

Skriv formene gjennom modellen med 128 skjulte nevroner.
Hva tilfører ReLU? Hvorfor har siste lag ti utganger, uansett størrelsen på det skjulte laget?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Skriv din forklaring her.


## Oppgave 2: Gjenbruk og forstå treningsløkken

Gjør treningskoden din fra notatbok 04 om til funksjonen
`train_model(model, train_loader, num_epochs=25, learning_rate=0.001)`.

Funksjonen skal opprette kryssentropitap og en ny Adam-optimizer, trene modellen
og returnere en liste med gjennomsnittlig treningstap per epoke. Antall
treningsbilder finnes med `len(train_loader.dataset)`.

Skriv ut tapet etter hver epoke.
Senere skal vi bruke denne funksjonen til å trene nettverket.

<details>
<summary>Hint</summary>

Flytt løkken inn i en funksjon. Opprett optimizer inne i funksjonen slik at den tilhører modellen som sendes inn.

</details>

In [ ]:
# Skriv din kode her.


## Oppgave 3: Samle evalueringsresultater

Skriv `evaluate_model(model, data_loader)` ved å tilpasse testløkken fra
notatbok 04. Returner en `dict` med:

- `accuracy`: antall riktige delt på antall bilder.
- `predictions`: alle predikerte klasser, samlet i én tensor.
- `labels`: alle riktige klasser, samlet i samme rekkefølge.

Bruk `eval()` og `no_grad()`. Funksjonen skal ikke oppdatere modellen.

<details>
<summary>Hint</summary>

Samle tensorer i to lister og bruk `torch.cat` til slutt. Bruk `argmax(dim=1)` for prediksjoner.

</details>


In [ ]:
# Skriv din kode her.


## Oppsett

Vi bruker **128 skjulte nevroner**, batchstørrelse 64, 25 epoker og Adam med læringsrate 0.001.
Hold disse innstillingene faste gjennom hovedløpet. Startvekter og rekkefølgen
på treningsbildene er tilfeldige, så resultatene kan variere mellom kjøringer.


In [ ]:
from time import perf_counter

from torch.utils.data import DataLoader

# Eksempel på tidtaking; bruk samme mønster rundt selve treningen.
start = perf_counter()
sum(range(10_000))
print(f"Eksempelberegningen tok {perf_counter() - start:.6f} sekunder")


## Oppgave 4: Tren og rapporter nettverket

Tren nettverket med **128 skjulte nevroner** og undersøk resultatet.
Bruk `create_model`, `train_model` og `evaluate_model` fra oppgave 1–3.

1. Opprett en ny modell og kall den `model`.
2. Opprett `train_loader` og `test_loader` med batchstørrelse 64. Stokk bare treningen.
3. Tren med `train_model` og lagre tapshistorikken i `history`.
   Mål tiden rundt selve treningen og lagre antall sekunder i `training_time`.
4. Evaluer med `evaluate_model` og lagre resultatet i `results`.
   Skriv ut sluttap, nøyaktighet og treningstid. Tegn treningskurven med aksetekster.

<details><summary>Hint</summary>

Bruk `perf_counter()` før og etter treningen.
`history` er en liste med tap per epoke. `results` er ordboken fra
`evaluate_model`, så testnøyaktigheten finnes i `results["accuracy"]`.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør cellen for å sjekke løsningen.


In [ ]:
assert len(history) == 25, "history skal inneholde ett tap per epoke, totalt 25."
assert np.isfinite(history).all(), "Alle tap i history skal være endelige tall, uten nan eller inf."
assert history[-1] < history[0], "Siste treningstap skal være lavere enn det første. Sjekk treningsløkken og parameteroppdateringen."
assert results["predictions"].shape == (1000,), "results[\"predictions\"] skal inneholde én prediksjon per testbilde, med formen (1000,)."
assert 0 <= results["accuracy"] <= 1, "results[\"accuracy\"] skal være en andel mellom 0 og 1."

print("Kontroll bestått!")


### Refleksjon

Oppgi første og siste treningstap, testnøyaktighet og tid fra din kjøring.
Hva kan vi konkludere med? Kan vi avgjøre om modellen overtilpasser bare ved å
se at treningstapet synker? Hvorfor bør vi ikke velge læringsrate ut fra testresultatet?

Sammenlign også testnøyaktigheten med den lineære modellen fra notatbok 04.
Hva observerer du? Er ett forsøk nok til å si at et nevralt nettverk alltid er bedre?


**Mitt svar:**

Skriv din forklaring her.


## Eksempel: finn posisjoner som oppfyller en betingelse

`torch.where` kan finne posisjonene der to tensorer er forskjellige.
Dette bruker vi til å finne feilklassifiserte bilder.


In [ ]:
example_predictions = torch.tensor([1, 2, 3, 4])
example_targets = torch.tensor([1, 7, 3, 9])
error_positions = torch.where(example_predictions != example_targets)[0]
print("Feil på posisjonene:", error_positions.tolist())


## Oppgave 5: Vis modellens feil

Bruk den forhåndsvalgte modellen med **128 skjulte nevroner**.
Finn feilindeksene fra `results` i oppgave 4 og vis opptil ni feilklassifiserte
testbilder. Skriv riktig og predikert klassenavn i tittelen for hvert bilde.
Bruk `CLASS_NAMES` til å slå opp navnene fra klasseindeksene.
Håndter også at modellen kan ha færre enn ni feil: skjul ubrukte akser.

`test_loader` stokker ikke, så resultatene har samme rekkefølge som `test_data`.

<details>
<summary>Hint</summary>

Kombiner `torch.where` med rutenettet fra notatbok 02. Bruk `index.item()` når du indekserer datasettet.

</details>


In [ ]:
# Skriv din kode her.


### Refleksjon

Beskriv to konkrete feil fra figuren. Ser noen av klassene like ut?
Er ni utvalgte feil nok til å si hvilken klasse modellen er dårligst på?
Hva måtte vi undersøkt for å svare bedre på det?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Skriv din forklaring her.


## Ferdig eksempel: vis sannsynligheter for ett bilde

Vi bruker den trente modellen `model` fra oppgave 4 og henter ett bilde fra `test_data`.

Softmax omformer skårene til positive tall som summerer til 1.
Dette er modellens beregnede sannsynligheter, ikke en garanti for at den har rett.
Vi bruker softmax til visning, aldri før kryssentropitapet.


In [ ]:
image, label = test_data[0]
model.eval()
with torch.no_grad():
    scores = model(image.unsqueeze(0))
    probabilities = scores.softmax(dim=1).squeeze(0)
fig, ax = plt.subplots()
ax.bar(range(10), probabilities.numpy())
ax.set_xticks(range(10), CLASS_NAMES, rotation=45, ha="right")
ax.set(xlabel="Klasse", ylabel="Beregnet sannsynlighet", title=f"Riktig klasse: {CLASS_NAMES[label]}")
fig.tight_layout()
plt.show()


## Selvstendig oppsummering – 45 minutter

Her skal du sette sammen delene med mindre støtte. Du kan slå opp tidligere
eksempler, men skriv koden og forklaringene selv. Arbeid gjerne to og to,
med hver deres skriftlige svar.


## Oppgave 6: Bygg og forklar hele kjeden

Lag en **ny** modell med `Flatten`, `Linear(784, 64)`, `ReLU` og `Linear(64, 10)`.
Kall den `own_model`. Opprett nye datalastere direkte med `DataLoader` (64 bilder per gruppe,
trening med stokking og test uten stokking) og
en ny Adam-optimizer med læringsrate 0.001. Tren modellen i én epoke og evaluer
den på testutvalget. Skriv både treningsløkken og evalueringsløkken direkte
i cellen; ikke kall `train_model` eller `evaluate_model`.

Skriv ut gjennomsnittlig treningstap og testnøyaktighet. Denne korte repetisjonen
er ikke en rettferdig ytelsessammenligning med modellene som fikk 25 epoker.

<details>
<summary>Hint</summary>

Skriv først rekkefølgen med egne ord: data → skårer → tap → gradienter → oppdatering. Legg så til evaluering uten oppdatering.

</details>


In [ ]:
# Skriv din kode her.


### Refleksjon

Forklar hele kjeden fra en gruppe bilder til oppdaterte parametere.
Ta med tensorformene, tapsfunksjonens rolle og forskjellen mellom gradientberegning
og parameteroppdatering. Forklar også hvorfor evaluering er en separat løkke.

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Skriv din forklaring her.


## Valgfri ekstraoppgave

Lag en forvekslingsmatrise for den ferdigtrente 128-modellen: rader er riktige
klasser, kolonner er predikerte klasser. Vis den med `imshow`, fargeskala og norske
aksetekster og klassenavn langs begge aksene. Undersøk hvilke forvekslinger som er vanligst. Bruk eksisterende
prediksjoner; ikke tren eller velg nye modeller ut fra testresultatet.


In [ ]:
# Skriv din kode her.


## Valgfri fordypning: 8 eller 128 skjulte nevroner?

Sammenlign de to forhåndsvalgte størrelsene. Du har allerede 128-resultatet.
Nå som vi skal sammenligne flere modeller, er det nyttig å samle resultatene
etter antall skjulte nevroner. Opprett ordbøkene `histories`, `training_times`
og `comparison_results` med nøkkelen `128` og verdiene fra `history`,
`training_time` og `results`.

Tren en ny 8-modell med samme batchstørrelse, læringsrate og antall epoker.
Lagre de nye verdiene i ordbøkene med nøkkelen `8`.
Skriv ut resultater og tegn treningskurvene i samme figur.

Ett forsøk kan ikke bevise at flere nevroner alltid er bedre. Ikke bruk
sammenligningen til videre tuning på testsettet.


In [ ]:
# Skriv din kode her.


### Refleksjon

Oppgi begge modellresultatene fra din kjøring. Hvilken praktisk forskjell ser du?
Kan vi konkludere med at den ene størrelsen alltid er bedre?


**Mitt svar:**

Skriv din forklaring her.


## Videre lesing

Lenkene er på engelsk og er valgfrie oppslag eller fordypning. Du trenger ikke lese alt for å løse oppgavene.

- **Repetisjon:** [3Blue1Brown – But what is a Neural Network?](https://www.3blue1brown.com/lessons/neural-networks/). Gå tilbake til forklaringen av skjulte lag etter at du har trent ditt eget nettverk. Knytt illustrasjonene til vektene, aktiveringene og formene i modellen din.
- **Kodeoppslag:** [PyTorch – Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html). Se spesielt hvordan `Linear` og `ReLU` settes sammen til et nettverk med skjulte lag.
- **Videre forståelse:** [Google Machine Learning Crash Course – Overfitting](https://developers.google.com/machine-learning/crash-course/overfitting/overfitting). Forklarer overtilpasning og generalisering. Bruk figurene til å diskutere hvorfor fallende treningstap alene ikke er nok til å vurdere modellen.
